# Étape 1 : Extraction du texte et des images

3.1. Définir le chemin du PDF

In [2]:
from pathlib import Path
import pymupdf

PDF_PATH = Path("./pdfs/attention is all you need.pdf")

OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Fichier introuvable : {PDF_PATH.resolve()}"
    )

TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF : {PDF_PATH.resolve()}")
print(f"Résultats : {OUTPUT_DIR.resolve()}")

PDF : C:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\pdfs\attention is all you need.pdf
Résultats : C:\Users\Lenovo\Downloads\RAG-Multi-Modal-Agentique\resultats_pdf


3.2. Extraire le texte et les images

In [3]:
textes = []
nombre_images = 0
pages_sans_texte = []

with pymupdf.open(PDF_PATH) as document:

    nombre_pages = len(document)

    for numero_page, page in enumerate(document, start=1):

        # 1. Extraction du texte
        texte = page.get_text("text", sort=True)

        # 2. Sauvegarde du texte de chaque page
        fichier_texte = TEXT_DIR / f"page_{numero_page:03d}.txt"

        fichier_texte.write_text(
            texte,
            encoding="utf-8"
        )

        textes.append(
            f"===== PAGE {numero_page} =====\n\n{texte}"
        )

        # 3. Extraction des images
        images_vues = set()

        for image_info in page.get_images(full=True):

            xref = image_info[0]

            if xref in images_vues:
                continue

            images_vues.add(xref)

            image = document.extract_image(xref)

            if not image:
                continue

            extension = image["ext"]

            fichier_image = (
                IMAGE_DIR /
                f"page_{numero_page:03d}_image_{xref}.{extension}"
            )

            fichier_image.write_bytes(image["image"])
            nombre_images += 1

# 4. Sauvegarder le texte intégral
(OUTPUT_DIR / "texte_complet.txt").write_text(
    "\n\n".join(textes),
    encoding="utf-8"
)

print("Pages traitées :", nombre_pages)
print("Images enregistrées :", nombre_images)

Pages traitées : 11
Images enregistrées : 3
